# EatSure Multilingual Complaint Analyzer — Training and Streamlit Deployment

This notebook trains multilingual complaint models from the supplied CSV, evaluates them on a held-out split, and creates a self-contained `deployment/` folder containing the Streamlit app, model bundle, aggregate-only dashboard data, requirements, and deployment notes.

**Core contribution retained:** character-level TF-IDF with classical machine learning for English, Hinglish, Hindi, Tamil, Telugu, and Kannada. English translation is an optional display aid; the category model still classifies the original complaint.

**Privacy:** raw customer rows and identifiers are not copied into deployment outputs. The app's dashboard uses grouped aggregate counts only. Keep the source CSV local and do not commit it.

Run top to bottom. If the CSV is not found automatically, set `DATA_PATH` in the next cell.


In [ ]:
from pathlib import Path
import re, unicodedata, json, warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 42
DATA_PATH = None  # Optional: r"C:\path\to\eatsure_complaints_feedback.csv"
OUTPUT_DIR = Path("deployment")
OUTPUT_DIR.mkdir(exist_ok=True)

possible_paths = [Path(DATA_PATH)] if DATA_PATH else []
possible_paths += [Path("eatsure_complaints_feedback.csv"), Path("../eatsure_complaints_feedback.csv"),
                   Path(r"C:\Users\ponss\Downloads\NLP Pro\eatsure_complaints_feedback.csv")]
csv_path = next((p for p in possible_paths if p and p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("CSV not found. Set DATA_PATH to eatsure_complaints_feedback.csv and rerun.")
df = pd.read_csv(csv_path, encoding="utf-8-sig")
required = {"complaint_text", "category"}
missing = required - set(df.columns)
if missing: raise ValueError(f"Missing required columns: {sorted(missing)}")
df = df.dropna(subset=["complaint_text", "category"]).copy()
df["complaint_text"] = df["complaint_text"].astype(str).str.strip()
df["category"] = df["category"].astype(str).str.strip()
df = df[(df.complaint_text != "") & (df.category != "")].reset_index(drop=True)
print(f"Loaded {len(df):,} usable complaints from {csv_path.name}")
print("Categories:", df.category.value_counts().to_dict())
if "language" in df: print("Languages:", df.language.value_counts().to_dict())
display(df[[c for c in ["complaint_text", "category", "sub_category", "language"] if c in df]].head())


## Prepare a leakage-resistant split

Repeated exact complaint texts are handled before splitting so identical text cannot appear in both training and test data. Texts with conflicting category labels are removed from model fitting and evaluation rather than arbitrarily assigning one label. The test portion remains untouched until final evaluation.


In [ ]:
def clean_text(text):
    text = unicodedata.normalize("NFC", str(text)).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\b(?:es|cmp|cust)\d+\b", " ID ", text, flags=re.I)
    text = re.sub(r"(?:₹|rs\.?|inr)\s*\d+(?:\.\d+)?", " AMOUNT ", text, flags=re.I)
    return re.sub(r"\s+", " ", text).strip()

df["clean_text"] = df.complaint_text.map(clean_text)
label_counts = df.groupby("clean_text")["category"].nunique()
conflicting = set(label_counts[label_counts > 1].index)
model_df = df[~df.clean_text.isin(conflicting)].drop_duplicates("clean_text").reset_index(drop=True)
if model_df.category.nunique() < 2: raise ValueError("At least two categories are needed.")
train_df, test_df = train_test_split(model_df, test_size=0.20, random_state=RANDOM_STATE, stratify=model_df.category)
print(f"Raw usable rows: {len(df):,}; conflicting texts excluded: {len(conflicting):,}; unique model rows: {len(model_df):,}")
print(f"Train rows: {len(train_df):,}; held-out test rows: {len(test_df):,}")


## Train and evaluate text models

The notebook compares Logistic Regression and Multinomial Naive Bayes on the same character n-gram features, selects by held-out macro-F1, then refits the selected classifier on all eligible category examples for deployment. Additional heads predict subcategory, sentiment, and priority when those columns have enough examples. They are advisory outputs and should be reviewed by a human.


In [ ]:
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
X_train = vectorizer.fit_transform(train_df.clean_text)
X_test = vectorizer.transform(test_df.clean_text)
models = {
    "Logistic Regression": LogisticRegression(max_iter=1200, class_weight="balanced", random_state=RANDOM_STATE),
    "Multinomial Naive Bayes": MultinomialNB()
}
rows=[]; fitted={}
for name, clf in models.items():
    clf.fit(X_train, train_df.category)
    pred=clf.predict(X_test)
    rows.append({"Model":name,"Accuracy":accuracy_score(test_df.category,pred),"Macro-F1":f1_score(test_df.category,pred,average="macro"),"Weighted-F1":f1_score(test_df.category,pred,average="weighted")})
    fitted[name]=clf
model_comparison=pd.DataFrame(rows).sort_values("Macro-F1",ascending=False).reset_index(drop=True)
display(model_comparison)
best_name=str(model_comparison.iloc[0]["Model"])
print("Selected model:",best_name)
category_test_pred=fitted[best_name].predict(X_test)
print(classification_report(test_df.category,category_test_pred,zero_division=0))
print("Confusion matrix labels:", sorted(model_df.category.unique()))
display(pd.DataFrame(confusion_matrix(test_df.category,category_test_pred,labels=sorted(model_df.category.unique())), index=sorted(model_df.category.unique()), columns=sorted(model_df.category.unique())))

# Refit final category model on all eligible examples.
final_vectorizer=TfidfVectorizer(analyzer="char_wb",ngram_range=(2,5),min_df=2,sublinear_tf=True)
X_all=final_vectorizer.fit_transform(model_df.clean_text)
final_category=LogisticRegression(max_iter=1200,class_weight="balanced",random_state=RANDOM_STATE) if best_name=="Logistic Regression" else MultinomialNB()
final_category.fit(X_all,model_df.category)

def fit_optional_head(column):
    if column not in df.columns: return None, None
    part=df.dropna(subset=["complaint_text",column]).copy()
    part[column]=part[column].astype(str).str.strip()
    part=part[(part[column]!="") & (part[column].str.lower()!="nan")]
    # Avoid fitting unstable heads on very rare classes.
    counts=part[column].value_counts()
    part=part[part[column].isin(counts[counts>=8].index)]
    if len(part)<100 or part[column].nunique()<2: return None, None
    head=LogisticRegression(max_iter=1200,class_weight="balanced",random_state=RANDOM_STATE)
    head.fit(final_vectorizer.transform(part.complaint_text.map(clean_text)),part[column])
    return head, sorted(part[column].unique().tolist())

subcategory_model,subcategory_labels=fit_optional_head("sub_category")
sentiment_model,sentiment_labels=fit_optional_head("sentiment")
priority_model,priority_labels=fit_optional_head("priority")
print("Optional heads:", {"subcategory": bool(subcategory_model), "sentiment": bool(sentiment_model), "priority": bool(priority_model)})
model_bundle={"vectorizer":final_vectorizer,"category_model":final_category,"subcategory_model":subcategory_model,"sentiment_model":sentiment_model,"priority_model":priority_model,"labels":{"category":sorted(model_df.category.unique().tolist()),"subcategory":subcategory_labels,"sentiment":sentiment_labels,"priority":priority_labels},"metadata":{"selected_model":best_name,"languages":sorted(df.language.dropna().astype(str).unique().tolist()) if "language" in df else [],"test_accuracy":float(model_comparison.iloc[0].Accuracy),"test_macro_f1":float(model_comparison.iloc[0]["Macro-F1"])}}
joblib.dump(model_bundle,OUTPUT_DIR/"eat_sure_models.joblib",compress=3)
print("Saved model bundle:",OUTPUT_DIR/"eat_sure_models.joblib")


## Create privacy-preserving dashboard aggregates

Only grouped counts are exported. Complaint text, customer IDs, order IDs, agent IDs, timestamps, and other row-level records are not placed in the deployment folder.


In [ ]:
# Dashboard export for public hosting: only coarsened group summaries, never row-level complaint records.
agg_df=df.copy()
if "complaint_timestamp" in agg_df:
    dates=pd.to_datetime(agg_df.complaint_timestamp,errors="coerce")
    agg_df["month"]=dates.dt.to_period("M").astype(str).replace("NaT","Unknown")
else:
    agg_df["month"]="Unknown"
for col in ["category","language","sub_category","priority","sentiment","resolution_status","channel"]:
    if col not in agg_df: agg_df[col]="Unknown"
    agg_df[col]=agg_df[col].fillna("Unknown").astype(str).replace("","Unknown")
core=agg_df.groupby(["month","category","language"],dropna=False).size().rename("complaint_count").reset_index()
sub=agg_df.groupby(["category","sub_category"],dropna=False).size().rename("complaint_count").reset_index()
parts=[]
for dim in ["priority","sentiment","resolution_status","channel"]:
    one=agg_df.groupby(dim,dropna=False).size().rename("complaint_count").reset_index().rename(columns={dim:"value"})
    one.insert(0,"dimension",dim)
    parts.append(one)
marginals=pd.concat(parts,ignore_index=True)
# Small groups are suppressed to protect sparse combinations in the public dashboard.
core=core[core.complaint_count>=5].reset_index(drop=True)
sub=sub[sub.complaint_count>=5].reset_index(drop=True)
marginals=marginals[marginals.complaint_count>=5].reset_index(drop=True)
core.to_csv(OUTPUT_DIR/"dashboard_aggregates.csv",index=False,encoding="utf-8")
sub.to_csv(OUTPUT_DIR/"dashboard_subcategories.csv",index=False,encoding="utf-8")
marginals.to_csv(OUTPUT_DIR/"dashboard_marginals.csv",index=False,encoding="utf-8")
print("Created coarsened dashboard summaries; minimum published group count:",min(core.complaint_count.min(),sub.complaint_count.min(),marginals.complaint_count.min()))


## Generate the Streamlit application and deployment files

The app provides guided single-complaint analysis, optional English translation, batch CSV analysis, interactive aggregate dashboards, model information, confidence-based review flags, and downloadable results. Translation uses an optional online Google translation service through `deep-translator`; if the service is unavailable, classification still works and the UI reports the translation issue.


In [ ]:
APP_SOURCE = 'from pathlib import Path\nimport re, unicodedata\nimport joblib\nimport pandas as pd\nimport streamlit as st\nimport plotly.express as px\n\nst.set_page_config(page_title="EatSure Complaint Intelligence", page_icon="🍽️", layout="wide", initial_sidebar_state="expanded")\n\nst.markdown("""\n<style>\n:root { color-scheme: light !important; }\n.stApp { background: #f6f9fc !important; color: #183247 !important; --text-color: #183247; --background-color: #f6f9fc; --secondary-background-color: #ffffff; --primary-color: #147a58; }\n[data-testid="stAppViewContainer"] { background: #f6f9fc !important; color: #183247 !important; }\n[data-testid="stHeader"] { background: rgba(246,249,252,.96) !important; }\n[data-testid="stSidebar"] { background: linear-gradient(180deg,#e9f7f1,#edf5ff) !important; border-right: 1px solid #d9e6e9; }\n[data-testid="stSidebar"] * { color: #183247 !important; opacity: 1 !important; }\n[data-testid="stAppViewContainer"] p, [data-testid="stAppViewContainer"] label, [data-testid="stAppViewContainer"] span, [data-testid="stAppViewContainer"] [data-testid="stMarkdownContainer"], [data-testid="stAppViewContainer"] [data-testid="stMetricLabel"], [data-testid="stAppViewContainer"] [data-testid="stMetricValue"], [data-testid="stAppViewContainer"] [data-testid="stWidgetLabel"] { color: #183247 !important; opacity: 1 !important; }\n[data-testid="stMetric"] { background: #ffffff !important; border: 1px solid #d7e3eb !important; padding: .85rem !important; border-radius: 14px !important; }\n[data-testid="stMetricLabel"] p { color: #496174 !important; }\n[data-testid="stMetricValue"] { color: #123b33 !important; font-weight: 700 !important; }\n[data-testid="stAlert"] * { color: #174a68 !important; opacity: 1 !important; }\n[data-testid="stAppViewContainer"] a { color: #075f9b !important; }\n[data-testid="stAppViewContainer"] button[kind="primary"] { background: #147a58 !important; border-color: #147a58 !important; }\n[data-testid="stAppViewContainer"] button[kind="primary"] *, [data-testid="stAppViewContainer"] button[kind="primary"] p { color: #ffffff !important; }\n.hero { background: linear-gradient(110deg,#e7f7ef,#eaf3ff); padding: 1.35rem 1.6rem; border:1px solid #d7e9e1; border-radius:18px; margin-bottom:1rem; }\n.hero h1 { color:#123b33 !important; margin:0; }\n.hero p { color:#31536a !important; margin:.35rem 0 0; }\n.result { background:#ffffff; color:#183247; border:1px solid #dce8ee; border-left:6px solid #15966a; border-radius:14px; padding:1rem 1.2rem; }\n.result * { color:#183247 !important; opacity:1 !important; }\n.small-note { color:#496174; font-size:.9rem; }\n\n/* Keep form controls and open menus readable even when the viewer prefers dark mode. */\n[data-testid="stSelectbox"] [role="group"] { background-color: #ffffff !important; border: 1px solid #b8c9d5 !important; border-radius: .5rem !important; }\n[data-testid="stSelectbox"] [role="combobox"] { background: transparent !important; color: #183247 !important; -webkit-text-fill-color: #183247 !important; }\n[data-testid="stSelectbox"] [role="group"] svg { color: #52697a !important; }\n[data-testid="stCheckbox"] label > div:first-of-type { background-color: #ffffff !important; border: 1px solid #52697a !important; }\n[data-testid="stCheckbox"] label:has(input:checked) > div:first-of-type { background-color: #147a58 !important; border-color: #147a58 !important; }\n\n[data-testid="stTextArea"] textarea, [data-testid="stTextInput"] input, [data-testid="stNumberInput"] input, [data-baseweb="select"] > div, [data-baseweb="select"] input { background-color: #ffffff !important; color: #183247 !important; -webkit-text-fill-color: #183247 !important; border-color: #b8c9d5 !important; }\n[data-testid="stTextArea"] textarea::placeholder, [data-testid="stTextInput"] input::placeholder { color: #647b8d !important; -webkit-text-fill-color: #647b8d !important; }\n[data-baseweb="popover"], [data-baseweb="popover"] > div, [data-baseweb="menu"], [role="listbox"], [role="option"] { background-color: #ffffff !important; color: #183247 !important; }\n[role="option"]:hover, [aria-selected="true"][role="option"] { background-color: #e6f3ed !important; color: #123b33 !important; }\n[data-testid="stCheckbox"] [role="checkbox"] { background-color: #ffffff !important; border: 1px solid #52697a !important; }\n[data-testid="stCheckbox"] [role="checkbox"][aria-checked="true"] { background-color: #147a58 !important; border-color: #147a58 !important; }\n[data-testid="stCheckbox"] [role="checkbox"][aria-checked="true"] svg { color: #ffffff !important; fill: #ffffff !important; }\ninput[type="checkbox"] { accent-color: #147a58 !important; }\nbutton[kind="primary"], [data-testid="stFormSubmitButton"] button { background: #147a58 !important; border-color: #147a58 !important; }\nbutton[kind="primary"] *, [data-testid="stFormSubmitButton"] button * { color: #ffffff !important; }\n</style>\n""", unsafe_allow_html=True)\n\n@st.cache_resource\ndef load_bundle(): return joblib.load(Path(__file__).parent / "eat_sure_models.joblib")\n@st.cache_data\ndef load_aggregates():\n    p=Path(__file__).parent/"dashboard_aggregates.csv"\n    return pd.read_csv(p) if p.exists() else pd.DataFrame()\n\n@st.cache_data\ndef load_subcategories():\n    p=Path(__file__).parent/"dashboard_subcategories.csv"\n    return pd.read_csv(p) if p.exists() else pd.DataFrame()\n\n@st.cache_data\ndef load_marginals():\n    p=Path(__file__).parent/"dashboard_marginals.csv"\n    return pd.read_csv(p) if p.exists() else pd.DataFrame()\n\nbundle=load_bundle(); vectorizer=bundle["vectorizer"]\nSUPPORTED={"english / hinglish","english","hinglish","hindi","tamil","telugu","kannada"}\nLANG_OPTIONS=["Auto detect","English","Hinglish","Hindi","Tamil","Telugu","Kannada","Malayalam","Bengali","Gujarati","Punjabi","Other / unsure"]\n\ndef clean_text(text):\n    text=unicodedata.normalize("NFC",str(text)).lower()\n    text=re.sub(r"https?://\\S+|www\\.\\S+"," URL ",text)\n    text=re.sub(r"\\b(?:es|cmp|cust)\\d+\\b"," ID ",text,flags=re.I)\n    text=re.sub(r"(?:₹|rs\\.?|inr)\\s*\\d+(?:\\.\\d+)?"," AMOUNT ",text,flags=re.I)\n    return re.sub(r"\\s+"," ",text).strip()\n\ndef script_language(text):\n    ranges=[("Malayalam",0x0D00,0x0D7F),("Tamil",0x0B80,0x0BFF),("Telugu",0x0C00,0x0C7F),("Kannada",0x0C80,0x0CFF),("Hindi",0x0900,0x097F),("Bengali",0x0980,0x09FF),("Gujarati",0x0A80,0x0AFF),("Punjabi",0x0A00,0x0A7F)]\n    counts={name:sum(lo<=ord(ch)<=hi for ch in str(text)) for name,lo,hi in ranges}\n    latin=sum(ch.isascii() and ch.isalpha() for ch in str(text))\n    name=max(counts,key=counts.get) if counts and max(counts.values(),default=0)>0 else ("English / Hinglish" if latin else "Unknown")\n    return name\n\ndef analyze(text, chosen="Auto detect", translate=False):\n    text=str(text).strip()\n    language=script_language(text) if chosen=="Auto detect" else chosen\n    if language not in LANG_OPTIONS: language="Other / unsure"\n    if language in {"Other / unsure","Malayalam","Bengali","Gujarati","Punjabi"} or (language=="Auto detect"):\n        return {"supported":False,"language":language,"category":"Manual review","confidence":None,"subcategory":"—","sentiment":"—","priority":"—","translated":"","note":"This language is outside the classifier\'s supported languages. Please route it for manual review."}\n    raw=vectorizer.transform([clean_text(text)])\n    model=bundle["category_model"]\n    probs=model.predict_proba(raw)[0] if hasattr(model,"predict_proba") else None\n    pred=str(model.classes_[int(probs.argmax())]) if probs is not None else str(model.predict(raw)[0])\n    conf=float(probs.max()) if probs is not None else None\n    def optional(key):\n        m=bundle.get(key+"_model")\n        return str(m.predict(raw)[0]) if m is not None else "Not available"\n    translated=""; note=""\n    if translate:\n        try:\n            from deep_translator import GoogleTranslator\n            code={"English":"en","Hinglish":"en","English / Hinglish":"en","Hindi":"hi","Tamil":"ta","Telugu":"te","Kannada":"kn"}.get(language)\n            translated=GoogleTranslator(source=code or "auto",target="en").translate(text) or ""\n        except Exception as exc:\n            note="Translation could not be completed right now. The original text was still analyzed."\n    return {"supported":True,"language":language,"category":pred,"confidence":conf,"subcategory":optional("subcategory"),"sentiment":optional("sentiment"),"priority":optional("priority"),"translated":translated,"note":note}\n\ndef confidence_label(conf):\n    if conf is None: return "Needs review"\n    return "High confidence" if conf>=.75 else "Review recommended" if conf>=.50 else "Manual review recommended"\n\nst.sidebar.title("EatSure Intelligence")\npage=st.sidebar.radio("Navigate",["Overview","Analyze complaint","Batch analysis","Insights","About the model"])\nst.sidebar.caption("Multilingual • code-mixed • character n-gram NLP")\n\nif page=="Overview":\n    st.markdown(\'<div class="hero"><h1>Complaint Intelligence</h1><p>Understand customer issues across English and Indian languages, then prioritize the next action.</p></div>\',unsafe_allow_html=True)\n    ag=load_aggregates()\n    cols=st.columns(4)\n    for col,label,value in zip(cols,["Categories","Supported languages","Model test accuracy","Test macro-F1"],[len(bundle["labels"]["category"]),"6",f\'{bundle["metadata"].get("test_accuracy",0):.1%}\',f\'{bundle["metadata"].get("test_macro_f1",0):.1%}\']): col.metric(label,value)\n    st.subheader("Start with a task")\n    a,b,c=st.columns(3)\n    a.info("**Analyze one complaint**\\n\\nDetect language, classify the issue, and review confidence.")\n    b.info("**Analyze a CSV**\\n\\nAdd model results to a file and download it for follow-up.")\n    c.info("**Explore patterns**\\n\\nFilter aggregate trends by category, language, priority, and channel.")\n    st.caption("Predictions are decision support. Confirm low-confidence or sensitive cases with a human reviewer.")\n\nelif page=="Analyze complaint":\n    st.markdown(\'<div class="hero"><h1>Analyze a complaint</h1><p>Enter the customer complaint. Language detection is script-based; you can override it below.</p></div>\',unsafe_allow_html=True)\n    with st.form("single"):\n        complaint=st.text_area("Customer complaint",height=150,placeholder="Example: मेरा पैसा कट गया लेकिन ऑर्डर नहीं हुआ।")\n        c1,c2=st.columns([1,1]); language=c1.selectbox("Language",LANG_OPTIONS); do_translate=c2.checkbox("Also translate to English",value=False)\n        submitted=st.form_submit_button("Analyze complaint",type="primary",use_container_width=True)\n    if submitted:\n        if not complaint.strip(): st.warning("Enter a complaint first.")\n        else:\n            r=analyze(complaint,language,do_translate)\n            if not r["supported"]: st.warning(r["note"])\n            else:\n                st.markdown(f\'<div class="result"><h3>Suggested category: {r["category"]}</h3><p>{confidence_label(r["confidence"])} · Detected/selected language: {r["language"]}</p></div>\',unsafe_allow_html=True)\n                m1,m2,m3=st.columns(3); m1.metric("Confidence",f\'{r["confidence"]:.1%}\' if r["confidence"] is not None else "N/A"); m2.metric("Suggested sentiment",r["sentiment"]); m3.metric("Suggested priority",r["priority"])\n                with st.expander("Analysis details",expanded=True):\n                    st.write("**Suggested subcategory:**",r["subcategory"])\n                    st.write("**Original complaint:**",complaint)\n                    if r["translated"]: st.write("**English translation:**",r["translated"])\n                    if r["note"]: st.info(r["note"])\n                    st.caption("Translation is an optional reading aid; classification uses the original complaint text.")\n                st.subheader("Help improve review quality")\n                with st.form("feedback"):\n                    corrected=st.selectbox("If the category is wrong, select the correct category",["No correction"]+bundle["labels"]["category"])\n                    st.text_area("Reviewer note (optional)")\n                    st.form_submit_button("Record feedback locally",help="Feedback is not uploaded or persisted by this demo app.")\n\nelif page=="Batch analysis":\n    st.markdown(\'<div class="hero"><h1>Batch analysis</h1><p>Upload a CSV containing a complaint text column, then download a copy with suggested model outputs.</p></div>\',unsafe_allow_html=True)\n    upload=st.file_uploader("Upload CSV",type=["csv"])\n    if upload:\n        try:\n            data=pd.read_csv(upload)\n            text_col=st.selectbox("Complaint text column",data.columns.tolist(),index=data.columns.tolist().index("complaint_text") if "complaint_text" in data.columns else 0)\n            lang_col=st.selectbox("Language column (optional)",["Auto detect"]+data.columns.tolist())\n            if st.button("Analyze uploaded complaints",type="primary"):\n                with st.spinner("Analyzing rows..."):\n                    results=[]\n                    for _,row in data.iterrows():\n                        selected="Auto detect"\n                        if lang_col!="Auto detect" and pd.notna(row[lang_col]):\n                            val=str(row[lang_col]).strip().lower()\n                            selected=next((x for x in LANG_OPTIONS if x.lower()==val),"Auto detect")\n                        r=analyze(row[text_col],selected,False)\n                        results.append({"detected_language":r["language"],"suggested_category":r["category"],"confidence":r["confidence"],"review_status":confidence_label(r["confidence"]),"suggested_subcategory":r["subcategory"],"suggested_sentiment":r["sentiment"],"suggested_priority":r["priority"]})\n                    result=pd.concat([data.reset_index(drop=True),pd.DataFrame(results)],axis=1)\n                    st.session_state["batch_result"]=result\n        except Exception as exc: st.error(f"Could not read or analyze this CSV: {exc}")\n    result=st.session_state.get("batch_result")\n    if result is not None:\n        st.success(f"Analyzed {len(result):,} rows")\n        st.dataframe(result.head(100),use_container_width=True,hide_index=True)\n        st.download_button("Download analyzed CSV",result.to_csv(index=False).encode("utf-8-sig"),"eatsure_analyzed_complaints.csv","text/csv",type="primary")\n\nelif page=="Insights":\n    st.markdown(\'<div class="hero"><h1>Complaint insights</h1><p>Explore coarser grouped patterns. No complaint-level records or customer identifiers are included.</p></div>\',unsafe_allow_html=True)\n    ag=load_aggregates(); subs=load_subcategories(); marg=load_marginals()\n    if ag.empty: st.info("No aggregate dashboard file found.")\n    else:\n        chosen={}\n        with st.expander("Filters",expanded=True):\n            grid=st.columns(3)\n            for i,col in enumerate(["month","category","language"]):\n                opts=sorted(ag[col].dropna().astype(str).unique().tolist())\n                chosen[col]=grid[i].multiselect(col.replace("_"," ").title(),opts,default=[])\n        view=ag.copy()\n        for col,vals in chosen.items():\n            if vals: view=view[view[col].astype(str).isin(vals)]\n        total=int(view.complaint_count.sum()) if not view.empty else 0\n        a,b,c=st.columns(3); a.metric("Complaints in selection",f"{total:,}"); b.metric("Categories represented",view.category.nunique() if total else 0); c.metric("Languages represented",view.language.nunique() if total else 0)\n        if not view.empty:\n            left,right=st.columns(2)\n            category=view.groupby("category",as_index=False).complaint_count.sum().sort_values("complaint_count",ascending=False)\n            lang=view.groupby("language",as_index=False).complaint_count.sum().sort_values("complaint_count",ascending=False)\n            left.plotly_chart(px.bar(category,x="category",y="complaint_count",color="category",title="Complaints by category",color_discrete_sequence=px.colors.qualitative.Set2),use_container_width=True)\n            right.plotly_chart(px.bar(lang,x="language",y="complaint_count",color="language",title="Complaints by language",color_discrete_sequence=px.colors.qualitative.Pastel),use_container_width=True)\n            c1,c2=st.columns(2)\n            if not subs.empty:\n                subview=subs.copy()\n                if chosen["category"]: subview=subview[subview.category.astype(str).isin(chosen["category"])]\n                c1.plotly_chart(px.sunburst(subview,path=["category","sub_category"],values="complaint_count",title="Category → subcategory (overall counts)"),use_container_width=True)\n            if not marg.empty:\n                priority=marg[marg.dimension=="priority"].sort_values("complaint_count",ascending=False)\n                c2.plotly_chart(px.bar(priority,x="value",y="complaint_count",color="value",title="Priority mix (overall counts)"),use_container_width=True)\n            sentiment=marg[marg.dimension=="sentiment"].sort_values("complaint_count",ascending=False)\n            st.plotly_chart(px.bar(sentiment,x="value",y="complaint_count",color="value",title="Sentiment mix (overall counts)"),use_container_width=True)\n            st.caption("Category and language charts respond to the filters. Subcategory, priority, and sentiment charts show overall grouped counts.")\n        else: st.info("No rows match these filters.")\n\nelse:\n    st.markdown(\'<div class="hero"><h1>About the model</h1><p>How this project turns multilingual complaint text into reviewable decision support.</p></div>\',unsafe_allow_html=True)\n    st.subheader("Model pipeline")\n    st.markdown("1. Normalize the complaint while preserving its script.\\n2. Represent it with character-level TF-IDF n-grams (2–5 characters).\\n3. Classify into the learned complaint category.\\n4. Show confidence and mark uncertain cases for human review.\\n5. Optionally display an English translation as a reading aid.")\n    st.subheader("Evaluation")\n    st.write(f"Selected model: **{bundle[\'metadata\'].get(\'selected_model\',\'Unknown\')}**")\n    a,b=st.columns(2); a.metric("Held-out accuracy",f"{bundle[\'metadata\'].get(\'test_accuracy\',0):.1%}"); b.metric("Held-out macro-F1",f"{bundle[\'metadata\'].get(\'test_macro_f1\',0):.1%}")\n    st.write("Category labels:",", ".join(bundle["labels"]["category"]))\n    st.caption("Language auto-detection relies mainly on writing script, so Latin-script English and Hinglish can be difficult to distinguish. Confirm or override the language selection. Translation and model outputs can be wrong; use human review for consequential decisions.")\n'

(OUTPUT_DIR / 'app.py').write_text(APP_SOURCE, encoding='utf-8')
(OUTPUT_DIR / "requirements.txt").write_text("""streamlit>=1.38
pandas>=2.0
numpy>=1.24
scikit-learn>=1.3
joblib>=1.3
plotly>=5.20
deep-translator>=1.11
""", encoding="utf-8")
(OUTPUT_DIR / "README.md").write_text("""# EatSure Complaint Intelligence

## Run locally

```bash
pip install -r requirements.txt
streamlit run app.py
```

Commit this folder to a GitHub repository, then create a Streamlit Community Cloud app pointing to `deployment/app.py`. The model bundle and aggregate dashboard CSV must remain beside `app.py`.

The notebook does not export raw complaint rows or identifiers. Translation is optional and needs an internet connection. Model predictions are advisory and uncertain cases should be reviewed by a person.
""", encoding="utf-8")
print('Deployment package created:')
for path in sorted(OUTPUT_DIR.iterdir()): print(f' - {path.name} ({path.stat().st_size:,} bytes)')

## After running the notebook

The `deployment/` folder is the app package to commit to GitHub. Check that it contains `app.py`, `eat_sure_models.joblib`, `dashboard_aggregates.csv`, `requirements.txt`, and `README.md`. Do not add the original CSV, notebook execution caches, or customer-level exports to a public repository. In Streamlit Community Cloud, select the repository and set the app entry point to `deployment/app.py`.
